# Electricity Theft Detection — Full Benchmark (Kaggle GPU)

Runs the complete 5-stage benchmark (Stage 1 XGBoost → Stage 5 transfer to Pakistan data) on GPU.

**Before running:**
1. Dataset `etd-repo` (the zipped project: code + config + data) is attached to this notebook.
2. Notebook Settings → **Accelerator: GPU T4 x2** and **Internet: On** (needed for pip).

**After running:** download `results_and_models.zip` from the Output panel and unzip it at the local repo root.

In [ ]:
# Cell 1 — dependencies + GPU check
!pip install -q imbalanced-learn openpyxl
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Enable GPU: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# Cell 2 — copy the read-only dataset to a writable working dir
import shutil, os
shutil.copytree('/kaggle/input/etd-repo', '/kaggle/working/run', dirs_exist_ok=True)
%cd /kaggle/working/run
print(os.listdir('.'))

In [ ]:
# Cell 3 — convert Maheen's Pakistan workbook into the Stage 5 target CSV.
# Expected end line: Class balance (0=normal, 1=theft): {0: 42, 1: 42}
!python -m src.experiments.prepare_pakistan_target --input pakistan_sgcc_format.xlsx

In [ ]:
# Cell 4 — the FULL benchmark: preprocess -> stage1 -> stage2 -> stage3 -> stage4 -> stage5
# If this cell dies mid-run, re-run Cell 2, then add e.g. --from_stage 3 to resume.
!python -m src.experiments.run_all_benchmark --config config/config.yaml \
    --target_csv data/raw/pakistan/pakistan_target.csv

In [ ]:
# Cell 5 — package trained weights + results for download
!zip -qr /kaggle/working/results_and_models.zip experiments_results models
print('Download: Output panel -> results_and_models.zip')

## Bring results back to the laptop
```bash
cd electricity-theft-detection
unzip -o ~/Downloads/results_and_models.zip
```
This merges `models/checkpoints/` (API/demo weights), `models/stage_checkpoints/` (per-stage models) and `experiments_results/benchmark_results.json` (the final table) into the project.